In [7]:
# =================================================================================================
# Raw data : NHANES 2003-2014 Demographics, Laboratory, and Questionnaire datasets
# Purpose : Create one unified participant-level NHANES dataset by aggregating all domains using SEQN.
# Specification : Each NHANES dataset has SEQN "Participant ID", which is the unique ID used for merge.
# Files Used :
#   neighbourhood_nhanes_demographics_2003_2014_final.csv
#   neighbourhood_nhanes_labs_2003_2014_appended.csv
#   neighbourhood_nhanes_questionnaire_2003_2014_appended.csv
# Data Dictionary : NHANES 2003-2014 documentation
# Data Availability Matrix : Generated in this notebook using domain presence flags
# Source Site : https://wwwn.cdc.gov/nchs/nhanes/
# Developer Notes : Full outer join is used so no SEQN is dropped from any file.
# File Creator : Neha Khairnar
# =================================================================================================

# 📦 Step 1: Imports
import pandas as pd
from pathlib import Path

# =================================================================================================
# 📂 Step 2: Define Paths
# =================================================================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = BASE_DIR / "data" / "processed_v2"
demo_file = DATA_DIR / "neighbourhood_nhanes_demographics_2003_2014_raw.csv"
labs_file = DATA_DIR / "neighbourhood_nhanes_labs_2003_2004_raw.csv"
questionnaire_file = DATA_DIR / "neighbourhood_nhanes_questionnaire_2003_2004_raw.csv"
Examination_file = DATA_DIR / "neighbourhood_nhanes_examination_2003_2004_raw.csv"
Dietary_file = DATA_DIR / "neighbourhood_nhanes_dietary_2003_2004_raw.csv"


# =================================================================================================
# 📥 Step 3: Load Files
# =================================================================================================

demo = pd.read_csv(demo_file)
labs = pd.read_csv(labs_file)
questionnaire = pd.read_csv(questionnaire_file)
Examination = pd.read_csv(Examination_file)
Dietary = pd.read_csv(Dietary_file)

print("📊 Loaded Files:")
print("Demographics:", demo.shape)
print("Labs:", labs.shape)
print("Questionnaire:", questionnaire.shape)
print("Examination:", Examination.shape)
print("Dietary:", Dietary.shape)

# =================================================================================================
# 🧹 Step 4: Standardize Column Names
# =================================================================================================
demo.columns = demo.columns.str.upper()
labs.columns = labs.columns.str.upper()
questionnaire.columns = questionnaire.columns.str.upper()
Examination.columns = Examination.columns.str.upper()
Dietary.columns = Dietary.columns.str.upper()

# =================================================================================================
# ✅ Step 5: Validate SEQN Exists
# =================================================================================================

for name, df_check in {
    "demographics": demo,
    "labs": labs,
    "questionnaire": questionnaire,
    "Examination":Examination,
    "Dietary":Dietary
}.items():
    if "SEQN" not in df_check.columns:
        raise ValueError(f"❌ SEQN column missing in {name} file")
    else:
        print(f"✅ SEQN found in {name}")

# =================================================================================================
# 📊 Step 6: File Structure Summary
# =================================================================================================
summary = []
for name, df_temp in {
    "demographics": demo,
    "labs": labs,
    "questionnaire": questionnaire,
    "Examination":Examination,
    "Dietary":Dietary
}.items():
    summary.append({
        "file": name,
        "rows": df_temp.shape[0],
        "columns": df_temp.shape[1],
        "unique_seqn": df_temp["SEQN"].nunique(),
        "duplicate_seqn_count": df_temp.duplicated(subset=["SEQN"]).sum(),
        "column_names": list(df_temp.columns)
    })

summary_df = pd.DataFrame(summary)
display(summary_df[["file", "rows", "columns", "unique_seqn", "duplicate_seqn_count"]])
# =================================================================================================
# 🧠 Step 7: Create Domain Presence Flags
# =================================================================================================
#demo["PRESENT_DEMOGRAPHICS"] = True
#labs["PRESENT_LABS"] = True
#questionnaire["PRESENT_QUESTIONNAIRE"] = True
# =================================================================================================
# 🔗 Step 8: Merge All Files Using FULL OUTER JOIN on SEQN
# =================================================================================================

final_df = (
    demo
    .merge(questionnaire, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_QUESTIONNAIRE"))
    .merge(labs, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_LAB"))
    .merge(Examination, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_EXAMINATION"))
    .merge(Dietary, on="SEQN", how="left", validate="one_to_one", suffixes=("", "_DIETARY"))
)
# =================================================================================================
# 👀 Step 10: Validate Final Unified Dataset
# =================================================================================================

print("📊 Final Unified Dataset Shape:", final_df.shape)
print("Unique SEQN Count:", final_df["SEQN"].nunique())
print("Duplicate SEQN Count:", final_df.duplicated(subset=["SEQN"]).sum())
display(final_df.head(20))

OUTPUT_DIR = BASE_DIR / "data" / "processed_v2"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "neighbourhood_nhanes_unified_2003_2004.csv"
final_df.to_csv(OUTPUT_PATH, index=False)
print("\n✅ Final unified NHANES dataset saved to:")
print(OUTPUT_PATH)

# =================================================================================================
# 💾 Step 14: Save Availability Summary
# =================================================================================================
# availability_summary_path = OUTPUT_DIR / "neighbourhood_nhanes_unified_2003_2014_availability_summary.csv"
# availability_summary.to_csv(availability_summary_path, index=False)

# print("\n✅ Availability summary saved to:")
# print(availability_summary_path)

C:\Users\User\AppData\Local\Temp\ipykernel_15556\1377442467.py:40: DtypeWarning: Columns (0: SMDUPCA) have mixed types. Specify dtype option on import or set low_memory=False.
  questionnaire = pd.read_csv(questionnaire_file)


📊 Loaded Files:
Demographics: (10122, 45)
Labs: (9179, 30)
Questionnaire: (10122, 441)
Examination: (9643, 33)
Dietary: (9643, 240)
✅ SEQN found in demographics
✅ SEQN found in labs
✅ SEQN found in questionnaire
✅ SEQN found in Examination
✅ SEQN found in Dietary


,file,rows,columns,unique_seqn,duplicate_seqn_count
0,demographics,10122,45,10122,0
1,labs,9179,30,9179,0
2,questionnaire,10122,441,10122,0
3,Examination,9643,33,9643,0
4,Dietary,9643,240,9643,0


📊 Final Unified Dataset Shape: (10122, 785)
Unique SEQN Count: 10122
Duplicate SEQN Count: 0


,SEQN,SDDSRVYR,RIDSTATR,RIDEXMON,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDAGEEX,RIDRETH1,RIDRETH2,...,DR2TP204,DR2TP205,DR2TP225,DR2TP226,DR2_300,DR2_320,DR2_330,DR2BWATR,DR2CWATR,DR2TWS
0,21005.0,3.0,2.0,1.0,1.0,1.900000e+01,232.0,233.0,4.0,2.0,...,1.790000e-01,5.397605e-79,1.400000e-02,1.900000e-02,2.0,5.180000e+02,5.180000e+02,5.397605e-79,5.397605e-79,1.0
1,21006.0,3.0,2.0,2.0,2.0,1.600000e+01,203.0,205.0,4.0,2.0,...,1.600000e-02,5.397605e-79,5.397605e-79,5.397605e-79,1.0,9.472000e+02,9.472000e+02,5.397605e-79,5.397605e-79,1.0
2,21007.0,3.0,2.0,1.0,2.0,1.400000e+01,172.0,172.0,3.0,1.0,...,2.900000e-02,1.000000e-02,5.397605e-79,6.000000e-03,2.0,5.920000e+02,5.397605e-79,5.920000e+02,5.397605e-79,NaN
3,21008.0,3.0,2.0,2.0,1.0,1.700000e+01,208.0,209.0,4.0,2.0,...,4.500000e-02,5.397605e-79,5.397605e-79,2.000000e-03,2.0,8.584000e+02,8.584000e+02,5.397605e-79,5.397605e-79,1.0
4,21009.0,3.0,2.0,2.0,1.0,5.500000e+01,671.0,672.0,3.0,1.0,...,1.920000e-01,4.000000e-03,5.397605e-79,1.900000e-02,2.0,6.216000e+02,6.216000e+02,5.397605e-79,5.397605e-79,1.0
5,21010.0,3.0,2.0,2.0,2.0,5.200000e+01,633.0,634.0,3.0,1.0,...,2.600000e-02,5.397605e-79,5.397605e-79,1.000000e-03,3.0,8.584000e+02,8.584000e+02,5.397605e-79,5.397605e-79,2.0
6,21011.0,3.0,2.0,1.0,1.0,5.397605e-79,3.0,4.0,1.0,3.0,...,NaN,NaN,NaN,NaN,2.0,7.400000e+00,7.400000e+00,5.397605e-79,5.397605e-79,1.0
7,21012.0,3.0,2.0,2.0,1.0,6.300000e+01,765.0,766.0,4.0,2.0,...,2.430000e-01,3.000000e-03,5.397605e-79,1.200000e-02,2.0,5.397605e-79,5.397605e-79,5.397605e-79,5.397605e-79,NaN
8,21013.0,3.0,2.0,1.0,2.0,1.300000e+01,163.0,164.0,4.0,2.0,...,3.300000e-02,5.397605e-79,5.397605e-79,1.000000e-03,2.0,3.552000e+02,5.397605e-79,3.552000e+02,5.397605e-79,NaN
9,21014.0,3.0,2.0,2.0,1.0,3.000000e+00,42.0,42.0,4.0,2.0,...,4.700000e-02,5.397605e-79,5.397605e-79,3.000000e-03,2.0,5.920000e+02,5.920000e+02,5.397605e-79,5.397605e-79,1.0



✅ Final unified NHANES dataset saved to:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_ML_Data_Predictor\Clinical_Neighbourhood_ML_Data_Predictor\data\processed_v2\neighbourhood_nhanes_unified_2003_2004.csv
